In [2]:
!pip install datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 17.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.1/194.1 kB 18.3 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2024.10.0
    Uninstalling fsspec-2024.10.0:
      Successfully uninstalled fsspec-2024.10.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2024.10.0 requires fsspec==2024.10.0, but you have fsspec 2024.9.0 which is incompatible.


In [3]:
!pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 3.6 MB/s eta 0:00:00


In [4]:
!pip install sacrebleu rouge_score

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 3.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.0/104.0 kB 7.0 MB/s eta 0:00:00
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24935 sha256=9493b25fad66ed132ab72575b457b6bfbbe81773ff38ccee74ee0fa8eefddb7c
  Stored in directory: /root/.cache/pip/wheels/5f/dd/89/461065a73be61a532ff8599a28e9beef17985c9e9c31e541b4
Successfully built rouge_score


In [5]:
!pip install wandb

In [6]:
import os

# Disable Weights & Biases logging
os.environ["WANDB_DISABLED"] = "true"

In [7]:
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    GenerationConfig,
)
from evaluate import load
from accelerate import Accelerator

In [8]:
# Function to process and fine-tune the model for translation
def train_and_evaluate_translation(model_name, train_file, test_file, source_lang, target_lang):
    """
    Trains and evaluates a translation model.

    Args:
        model_name (str): Hugging Face model name or path.
        train_file (str): Path to the training dataset.
        test_file (str): Path to the test dataset.
        source_lang (str): Source language column name in the dataset.
        target_lang (str): Target language column name in the dataset.
    Returns:
        dict: Evaluation results including BLEU and ROUGE scores.
    """
    # Load dataset
    data_files = {"train": train_file, "test": test_file}
    dataset = load_dataset("csv", data_files=data_files, column_names=[source_lang, target_lang])

    # Load tokenizer and model
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

    # Configure model
    model.config.bos_token_id = 1
    model.config.eos_token_id = 0
    model.config.decoder_start_token_id = model.config.bos_token_id

    # Set generation configuration
    generation_config = GenerationConfig(
        bos_token_id=model.config.bos_token_id,
        eos_token_id=model.config.eos_token_id,
        decoder_start_token_id=model.config.decoder_start_token_id,
        max_length=128,
        num_beams=5,
    )
    model.generation_config = generation_config

    # Ensure compatibility between tokenizer and model
    model.resize_token_embeddings(len(tokenizer))

    # Preprocessing function
    def preprocess_function(examples):
        inputs = examples[source_lang]
        targets = examples[target_lang]
        model_inputs = tokenizer(inputs, max_length=128, truncation=True, padding="max_length")
        with tokenizer.as_target_tokenizer():
            labels = tokenizer(targets, max_length=128, truncation=True, padding="max_length")
        model_inputs["labels"] = labels["input_ids"]
        return model_inputs

    # Preprocess datasets
    tokenized_datasets = dataset.map(preprocess_function, batched=True)

    # Define training arguments
    training_args = Seq2SeqTrainingArguments(
        output_dir=f"./results_{source_lang}_to_{target_lang}",
        eval_strategy="epoch",
        learning_rate=3e-5,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=8,
        weight_decay=0.01,
        save_total_limit=3,
        num_train_epochs=7,
        predict_with_generate=True,
        fp16=True,
        report_to="none",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
    )

    # Load metrics
    bleu = load("sacrebleu")
    rouge = load("rouge")

    # Compute metrics
    def compute_metrics(eval_preds):
        preds, labels = eval_preds
        decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
        decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
        bleu_score = bleu.compute(predictions=decoded_preds, references=[[ref] for ref in decoded_labels])
        rouge_score = rouge.compute(predictions=decoded_preds, references=decoded_labels)
        return {
            "bleu": bleu_score["score"],
            "rouge1": rouge_score["rouge1"],
            "rouge2": rouge_score["rouge2"],
            "rougeL": rouge_score["rougeL"],
            "rougeLsum": rouge_score["rougeLsum"],
        }

    # Initialize trainer
    trainer = Seq2SeqTrainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_datasets["train"],
        eval_dataset=tokenized_datasets["test"],
        tokenizer=tokenizer,
        compute_metrics=compute_metrics,
    )

    # Train the model
    trainer.train()

    # Evaluate the model
    eval_results = trainer.evaluate()
    print(f"Evaluation Results for {source_lang} to {target_lang}:", eval_results)

    return eval_results


In [9]:
# Train and evaluate for both directions
if __name__ == "__main__":
    model_name = "Helsinki-NLP/opus-mt-en-mul"
    train_file = "/content/train_set.csv"  # Replace with your training dataset
    test_file = "/content/test_set.csv"    # Replace with your test dataset

    # Kalamang to English
    print("\nTraining Kalamang to English...")
    kalamang_to_english_results = train_and_evaluate_translation(
        model_name=model_name,
        train_file=train_file,
        test_file=test_file,
        source_lang="Kalamang_Sentence",
        target_lang="English_Translation",
    )

    # English to Kalamang
    print("\nTraining English to Kalamang...")
    english_to_kalamang_results = train_and_evaluate_translation(
        model_name=model_name,
        train_file=train_file,
        test_file=test_file,
        source_lang="English_Translation",
        target_lang="Kalamang_Sentence",
    )


Training Kalamang to English...


Generating train split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/44.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

... (21 outputs left out)

Epoch,Training Loss,Validation Loss,Bleu,Rouge1,Rouge2,Rougel,Rougelsum
1,No log,0.418383,2.780796,0.206615,0.099927,0.198901,0.198899
2,No log,0.375901,4.699281,0.241684,0.111412,0.233074,0.234275
3,0.642000,0.357299,7.870221,0.251456,0.135273,0.245445,0.245441
4,0.642000,0.347178,7.107664,0.264072,0.137673,0.254312,0.256246
5,0.642000,0.339020,9.247596,0.306006,0.163363,0.286255,0.285003
6,0.303500,0.339351,10.085117,0.309293,0.163234,0.296382,0.296819
7,0.303500,0.339217,10.584720,0.317761,0.168985,0.300851,0.300444


Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (ht

Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.


Evaluation Results for English_Translation to Kalamang_Sentence: {'eval_loss': 0.3390199542045593, 'eval_bleu': 9.247595676235806, 'eval_rouge1': 0.3060055645746264, 'eval_rouge2': 0.16336260989844986, 'eval_rougeL': 0.2862548283122488, 'eval_rougeLsum': 0.2850029294997445, 'eval_runtime': 9.8666, 'eval_samples_per_second': 10.237, 'eval_steps_per_second': 1.318, 'epoch': 7.0}


In [10]:
# Train and evaluate for both directions
if __name__ == "__main__":
    model_name = "google-t5/t5-base"
    train_file = "/content/train_set.csv"  # Replace with your training dataset
    test_file = "/content/test_set.csv"    # Replace with your test dataset

    # Kalamang to English
    print("\nTraining Kalamang to English...")
    kalamang_to_english_results = train_and_evaluate_translation(
        model_name=model_name,
        train_file=train_file,
        test_file=test_file,
        source_lang="Kalamang_Sentence",
        target_lang="English_Translation",
    )

    # English to Kalamang
    print("\nTraining English to Kalamang...")
    english_to_kalamang_results = train_and_evaluate_translation(
        model_name=model_name,
        train_file=train_file,
        test_file=test_file,
        source_lang="English_Translation",
        target_lang="Kalamang_Sentence",
    )


Training Kalamang to English...


config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

/usr/local/lib/python3.10/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


model.safetensors:   0%|          | 0.00/892M [00:00<?, ?B/s]

... (14 outputs left out)

Epoch,Training Loss,Validation Loss,Bleu,Rouge1,Rouge2,Rougel,Rougelsum
1,No log,0.363448,0.400199,0.146763,0.052901,0.144302,0.143451
2,No log,0.321196,0.079254,0.060546,0.019822,0.060967,0.060768
3,1.001800,0.305968,1.055044,0.150183,0.056808,0.143400,0.144238
4,1.001800,0.297659,2.100070,0.178133,0.074867,0.173924,0.174569
5,1.001800,0.292621,1.923154,0.195835,0.071102,0.188331,0.189037
6,0.312500,0.290358,2.193678,0.186626,0.074859,0.181178,0.181819
7,0.312500,0.289458,2.466158,0.189070,0.076983,0.182584,0.183742


Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:0 for o

Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.


Evaluation Results for English_Translation to Kalamang_Sentence: {'eval_loss': 0.2894580662250519, 'eval_bleu': 2.466158058486608, 'eval_rouge1': 0.18907023666141748, 'eval_rouge2': 0.07698317360151036, 'eval_rougeL': 0.18258350579225507, 'eval_rougeLsum': 0.18374196987050517, 'eval_runtime': 28.8036, 'eval_samples_per_second': 3.507, 'eval_steps_per_second': 0.451, 'epoch': 7.0}
